# Sprint 002 — Retail Sales Data Understanding

## Objective

This notebook explores the raw retail sales dataset, identifies data-quality issues, validates important business assumptions, and verifies the cleaned and rejected datasets produced by the Sprint 002 data pipeline.

The analysis focuses on:

- dataset structure and data types
- missing values
- duplicate records
- invalid sales, price, promotion, and stock values
- category and product consistency
- store ID consistency
- date coverage
- validation of cleaned and rejected outputs

## 1. Setup and Load Raw Data

First, we import the required Python libraries, define the project path, locate the raw dataset, and load it into a pandas DataFrame for analysis.

In [3]:
from pathlib import Path

import pandas as pd
print(Path.cwd())
PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)


c:\Users\gpc00\Projects\retail-demand-forecasting-platform\notebooks
c:\Users\gpc00\Projects\retail-demand-forecasting-platform


In [4]:
RAW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Sprint_002_Retail_Sales_Data.csv"
)

print(RAW_FILE)

c:\Users\gpc00\Projects\retail-demand-forecasting-platform\data\raw\Sprint_002_Retail_Sales_Data.csv


In [5]:
print(RAW_FILE.exists())

True


In [6]:
data = pd.read_csv(RAW_FILE)

## 2. Dataset Overview

This section checks the basic structure of the raw dataset, including the number of rows and columns, available fields, data types, and a sample of the first few records.

In [7]:
print(data.shape)

(26230, 8)


In [8]:
print(data.columns)

Index(['transaction_date', 'store_id', 'product_id', 'category', 'units_sold',
       'unit_price', 'promotion_flag', 'stock_on_hand'],
      dtype='str')


In [9]:
print(data.dtypes)

transaction_date        str
store_id                str
product_id              str
category                str
units_sold          float64
unit_price          float64
promotion_flag        int64
stock_on_hand       float64
dtype: object


In [10]:
data.head()

,transaction_date,store_id,product_id,category,units_sold,unit_price,promotion_flag,stock_on_hand
0,2026-04-02,ST001,P005,Snacks,22.0,214.47,0,93.0
1,2025-09-27,ST003,P009,Household,20.0,114.67,0,40.0
2,2025-02-18,ST004,P011,Personal Care,4.0,248.13,0,71.0
3,2026-02-01,ST003,P002,Beverages,48.0,356.76,0,24.0
4,2026-04-03,ST001,P012,Personal Care,34.0,439.25,0,79.0


## 3. Raw Data Quality Checks

This section identifies the main data-quality issues in the raw dataset before any cleaning is applied.

The checks include:

- missing values
- exact duplicate rows
- negative `units_sold`
- zero or negative `unit_price`
- invalid `promotion_flag` values
- negative `stock_on_hand`
- unknown product IDs
- store ID whitespace inconsistencies
- invalid transaction dates

In [11]:
data.isna().sum()

transaction_date     0
store_id             0
product_id           0
category            12
units_sold          35
unit_price           0
promotion_flag       0
stock_on_hand       28
dtype: int64

In [12]:
data.duplicated().sum()

np.int64(22)

In [13]:
(data["units_sold"] < 0).sum()

np.int64(16)

In [14]:
(data["unit_price"] <= 0).sum()

np.int64(10)

In [15]:
data["promotion_flag"].value_counts()

promotion_flag
0    24092
1     2130
2        8
Name: count, dtype: int64

In [16]:
(data["product_id"] == "UNKNOWN").sum()

np.int64(6)

In [17]:
(
    data["store_id"]
    != data["store_id"].str.strip()
).sum()

np.int64(15)

In [18]:
parsed_dates = pd.to_datetime(
    data["transaction_date"],
    errors="coerce"
)

parsed_dates.isna().sum()

np.int64(0)

In [19]:
print(parsed_dates.min())
print(parsed_dates.max())

2025-01-01 00:00:00
2026-06-30 00:00:00


In [20]:
data["category"].value_counts(dropna=False)

category
Household        6553
Personal Care    6551
Snacks           6548
Beverages        6548
NaN                12
beverages           7
snacks              5
personal care       4
household           2
Name: count, dtype: int64

In [21]:
data["store_id"].nunique()

8

In [22]:
data["store_id"].str.strip().nunique()

4

## 4. Category, Store, and Product Consistency

This section checks whether categorical identifiers are internally consistent before cleaning.

The analysis includes:

- category label normalization
- category value counts
- store ID whitespace inconsistencies
- unique store counts before and after trimming
- product-to-category consistency
- verification that missing categories can be safely inferred from known product IDs

In [23]:
normalized_category = (
    data["category"]
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
)

normalized_category.nunique()

4

In [24]:
normalized_category.value_counts(dropna=False)

category
household        6555
personal care    6555
beverages        6555
snacks           6553
NaN                12
Name: count, dtype: int64

In [25]:
category_product_check = pd.DataFrame({
    "product_id": data["product_id"],
    "category_normalized": normalized_category
})

In [26]:
category_product_check.groupby("product_id")["category_normalized"].nunique()

product_id
P001       1
P002       1
P003       1
P004       1
P005       1
P006       1
P007       1
P008       1
P009       1
P010       1
P011       1
P012       1
UNKNOWN    4
Name: category_normalized, dtype: int64

In [27]:
(
    category_product_check[
        category_product_check["product_id"] != "UNKNOWN"
    ]
    .groupby("product_id")["category_normalized"]
    .first()
)

product_id
P001        beverages
P002        beverages
P003        beverages
P004           snacks
P005           snacks
P006           snacks
P007        household
P008        household
P009        household
P010    personal care
P011    personal care
P012    personal care
Name: category_normalized, dtype: str

In [28]:
(
    data.loc[
        data["category"].isna(),
        "product_id"
    ]
    == "UNKNOWN"
).sum()

np.int64(0)

## 5. Detailed Anomaly Investigation

This section investigates the identified data-quality issues in more detail to determine whether the affected rows can be safely corrected or should be rejected.

The investigation covers:

- distribution of missing `units_sold` across stores and categories
- distribution of missing `stock_on_hand` across stores and categories
- overlap between missing-value rows
- exact duplicate pairs
- distribution of negative `units_sold`
- zero-price records and their promotion status
- invalid `promotion_flag` values
- `UNKNOWN` product records
- verification that quantity and stock values are whole numbers

In [29]:
(
    data.loc[
        data["units_sold"].isna(),
        "store_id"
    ]
    .str.strip()
    .value_counts()
)

store_id
ST002    10
ST003     9
ST004     9
ST001     7
Name: count, dtype: int64

In [30]:
(
    data.loc[
        data["units_sold"].isna(),
        "category"
    ]
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .value_counts()
)

category
beverages        10
snacks           10
personal care     9
household         6
Name: count, dtype: int64

In [31]:
(
    data.loc[
        data["stock_on_hand"].isna(),
        "store_id"
    ]
    .str.strip()
    .value_counts()
)

store_id
ST003    8
ST004    8
ST001    7
ST002    5
Name: count, dtype: int64

In [32]:
(
    data.loc[
        data["stock_on_hand"].isna(),
        "category"
    ]
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .value_counts()
)

category
household        8
beverages        8
personal care    7
snacks           5
Name: count, dtype: int64

In [33]:
missing_stock_percentage = (
    data["stock_on_hand"].isna().sum()
    / len(data)
) * 100

print(missing_stock_percentage)

0.1067479984750286


In [34]:
(data.isna().sum(axis=1) > 1).sum()

np.int64(0)

In [35]:
data.isna().any(axis=1).sum()

np.int64(75)

In [36]:
data[data.duplicated(keep=False)].shape[0]

44

In [37]:
data.duplicated(
    subset=["transaction_date", "store_id", "product_id"]
).sum()

np.int64(22)

In [38]:
data.loc[
    data["units_sold"] < 0,
    "units_sold"
].value_counts().sort_index()

units_sold
-7.0    3
-6.0    4
-5.0    2
-4.0    2
-2.0    3
-1.0    2
Name: count, dtype: int64

In [39]:
data.loc[
    data["unit_price"] <= 0,
    "unit_price"
].value_counts().sort_index()

unit_price
0.0    10
Name: count, dtype: int64

In [40]:
data.loc[
    data["unit_price"] == 0,
    "promotion_flag"
].value_counts()

promotion_flag
0    9
1    1
Name: count, dtype: int64

In [41]:
data.loc[
    ~data["promotion_flag"].isin([0, 1]),
    "promotion_flag"
].value_counts()

promotion_flag
2    8
Name: count, dtype: int64

In [42]:
data.loc[
    data["product_id"] == "UNKNOWN",
    ["transaction_date", "store_id", "category", "units_sold", "unit_price"]
]

,transaction_date,store_id,category,units_sold,unit_price
5534,2025-04-02,ST001,Snacks,21.0,213.71
11289,2025-07-28,ST004,Personal Care,38.0,440.43
11692,2025-07-30,ST003,Household,27.0,103.27
16562,2026-05-02,ST002,Personal Care,38.0,449.11
18900,2026-05-09,ST002,Household,41.0,66.32
26209,2025-05-13,ST001,Beverages,39.0,66.84


In [43]:
(
    data["units_sold"]
    .dropna()
    % 1
    != 0
).sum()

np.int64(0)

In [44]:
(
    data["stock_on_hand"]
    .dropna()
    % 1
    != 0
).sum()

np.int64(0)

## 6. Cleaned Dataset Validation

This section verifies that the cleaned dataset produced by the data pipeline satisfies the expected quality rules.

The validation checks include:

- expected row and column counts
- no missing values
- no duplicate rows
- no negative `units_sold`
- no zero or negative `unit_price`
- no negative `stock_on_hand`
- valid `promotion_flag` values only
- no `UNKNOWN` product IDs
- expected number of products, categories, and stores
- correct numeric data types
- valid transaction date range

In [45]:
PROCESSED_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "retail_sales_cleaned.csv"
)

cleaned_data = pd.read_csv(PROCESSED_FILE)

cleaned_data.shape

(26105, 8)

In [46]:
cleaned_data.isna().sum()

transaction_date    0
store_id            0
product_id          0
category            0
units_sold          0
unit_price          0
promotion_flag      0
stock_on_hand       0
dtype: int64

In [47]:
cleaned_data.duplicated().sum()

np.int64(0)

In [48]:
(cleaned_data["units_sold"] < 0).sum()

np.int64(0)

In [49]:
(cleaned_data["unit_price"] <= 0).sum()

np.int64(0)

In [50]:
(cleaned_data["stock_on_hand"] < 0).sum()

np.int64(0)

In [51]:
~cleaned_data["promotion_flag"].isin([0, 1])

0        False
1        False
2        False
3        False
4        False
         ...  
26100    False
26101    False
26102    False
26103    False
26104    False
Name: promotion_flag, Length: 26105, dtype: bool

In [52]:
(
    ~cleaned_data["promotion_flag"].isin([0, 1])
).sum()

np.int64(0)

In [53]:
(cleaned_data["product_id"] == "UNKNOWN").sum()

np.int64(0)

In [54]:
cleaned_data["product_id"].nunique()

12

In [55]:
cleaned_data["category"].nunique()

4

In [56]:
cleaned_data["store_id"].nunique()

4

In [57]:
cleaned_data.dtypes

transaction_date        str
store_id                str
product_id              str
category                str
units_sold            int64
unit_price          float64
promotion_flag        int64
stock_on_hand         int64
dtype: object

In [58]:
cleaned_data["transaction_date"] = pd.to_datetime(
    cleaned_data["transaction_date"]
)

cleaned_data.dtypes

transaction_date    datetime64[us]
store_id                       str
product_id                     str
category                       str
units_sold                   int64
unit_price                 float64
promotion_flag               int64
stock_on_hand                int64
dtype: object

In [59]:
print(cleaned_data["transaction_date"].min())
print(cleaned_data["transaction_date"].max())

2025-01-01 00:00:00
2026-06-30 00:00:00


## 7. Rejected Dataset Validation

This section verifies that rows removed from the cleaned dataset were preserved separately with a clear rejection reason.

The checks include:

- expected rejected row count
- presence of the `rejection_reason` column
- rejection counts by reason
- confirmation that all rejection reasons sum to the total rejected rows
- reconciliation of raw, cleaned, and rejected row counts

In [60]:
REJECTED_FILE = (
    PROJECT_ROOT
    / "data"
    / "rejected"
    / "retail_sales_rejected.csv"
)

rejected_data = pd.read_csv(REJECTED_FILE)

rejected_data.shape

(125, 9)

In [61]:
rejected_data["rejection_reason"].value_counts()

rejection_reason
missing units_sold                                          35
missing stock_on_hand                                       28
exact duplicate row                                         22
negative units_sold - business meaning unclear              16
unit_price is zero - business meaning unclear               10
promotion_flag is 2 - business meaning unclear               8
product_id is UNKNOWN - cannot identify product reliably     6
Name: count, dtype: int64

In [62]:
rejected_data["rejection_reason"].value_counts().sum()

np.int64(125)

In [63]:
len(cleaned_data) + len(rejected_data) == len(data)

True

## 8. Summary and Cleaning Decisions

The raw retail sales dataset contained 26,230 rows and 8 columns.

The analysis identified the following data-quality issues:

- 75 rows contained missing values
- 22 extra exact duplicate rows were present
- 16 rows contained negative `units_sold`
- 10 rows contained a zero `unit_price`
- 8 rows contained an invalid `promotion_flag` value of `2`
- 6 rows contained an `UNKNOWN` product ID
- store IDs contained whitespace inconsistencies
- category labels contained inconsistent formatting

The missing `category` values were safely recovered because all affected rows belonged to known products with a consistent product-to-category relationship.

Rows with missing or ambiguous business-critical values were not guessed or imputed without evidence. Instead, they were preserved in the rejected dataset with a clear rejection reason.

After cleaning:

- 26,105 rows remained in the cleaned dataset
- 125 rows were preserved in the rejected dataset
- no missing values remained
- no exact duplicate rows remained
- no invalid sales, price, promotion, stock, or product values remained
- 12 valid products remained
- 4 normalized categories remained
- 4 valid stores remained
- the full date range from 2025-01-01 to 2026-06-30 was preserved

The final reconciliation confirmed that:

**26,105 cleaned rows + 125 rejected rows = 26,230 original rows**

This verifies that the cleaning pipeline did not accidentally lose or create records.